# Rarest signatures: DPR and GMI swaths at peaks

Step 3 of 3. For each observation selected in step 2, downloads the 2A-DPR, 1C-GMI-R and 2A-GMI granules around its time, runs the PESCA retrieval and maps them with the nearby rarest observations.

Input: `final_rarest_4_nbr_count_0_m_3_peaks_to_analyze.parquet` (step 2) and `final_rarest.parquet`.
Output: maps and `peaks.log` in the images directory.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime
import logging
import pathlib

import cartopy.crs as ccrs
import gpm
import matplotlib.pyplot as plt
import polars as pl

from pmw_analysis.constants import ArgSurfaceType
from pmw_analysis.constants import DIR_PMW_ANALYSIS, ArgTransform, COLUMN_TIME, COLUMN_LON, COLUMN_LAT
from pmw_analysis.constants import DIR_IMAGES
from pmw_analysis.processing.filter import filter_by_value_range
from pmw_analysis.utils.io import combine_paths, file_to_dir

## Helpers

In [ ]:
def get_pd_col(freq: int) -> str:
    return f"PD_{freq}"

def get_diff_col(tc_min: str, tc_sub: str) -> str:
    return f"{tc_min}_{tc_sub}"

def add_pd_var(ds, freq: int):
    v = f"Tc_{freq}V"
    h = f"Tc_{freq}H"
    return ds.assign({get_pd_col(freq): ds[v] - ds[h]})


def add_diff_var(ds, tc_min: str, tc_sub: str):
    return ds.assign({get_diff_col(tc_min, tc_sub): ds[tc_min] - ds[tc_sub]})

def transform_ds(ds, drop=False):
    initial_vars = ds.data_vars
    # PD variables
    ds = add_pd_var(ds, 19)
    ds = add_pd_var(ds, 37)

    # difference variables
    ds = add_diff_var(ds, "Tc_37V", "Tc_19V")
    ds = add_diff_var(ds, "Tc_89V", "Tc_37V")

    # dropping (same logic as your Polars code)
    if drop:
        keep = ["Tc_37V"]
        ds = ds.drop_vars([v for v in initial_vars if v not in keep])

    return ds

In [ ]:
def _get_projection(peak: pl.DataFrame):
    if peak[COLUMN_LAT].item() > 60:
        projection = ccrs.NorthPolarStereo(central_longitude=peak[COLUMN_LON].item())
        figsize = (7, 4)
    elif peak[COLUMN_LAT].item() < -60:
        projection = ccrs.SouthPolarStereo(central_longitude=peak[COLUMN_LON].item())
        figsize = (7, 4)
    else:
        projection = ccrs.Mercator()
        figsize = (4, 4)

    return projection, figsize

## Settings and inputs

In [ ]:
arg_transform = ArgTransform.V4
arg_surface_type = ArgSurfaceType.LAND
df_dir_path = pathlib.Path(DIR_PMW_ANALYSIS) / arg_transform.value / arg_surface_type.value

In [ ]:
df_path = df_dir_path / "final_rarest_4_nbr_count_0_m_3_peaks_to_analyze.parquet"
df_peaks = pl.read_parquet(df_path)

In [ ]:
df_peaks = df_peaks.sort("lon", descending=True)

In [ ]:
peaks_indices = set(range(len(df_peaks)))

In [ ]:
df_peaks

In [ ]:
df_rarest_path = df_dir_path / "final_rarest.parquet"
df_rarest = pl.read_parquet(df_rarest_path)

## Output directory and logging

In [ ]:
images_dir = combine_paths(path_base=pathlib.Path("..") / DIR_IMAGES, path_rel=file_to_dir(df_path), path_rel_base=DIR_PMW_ANALYSIS)
images_dir = images_dir / "peaks"
images_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    handlers=[
        logging.FileHandler(images_dir / "peaks.log", mode="w"),
        logging.StreamHandler()
    ]
)

## Rarest observations within ±1 min of each peak

In [ ]:
df_rarest_list = []
for i in peaks_indices:
    peak = df_peaks[i]

    # Specify the time period you are interested in
    timedelta = datetime.timedelta(minutes=1)
    start_time = peak[COLUMN_TIME].item() - timedelta
    end_time = peak[COLUMN_TIME].item() + timedelta

    df_rarest_around_peak = filter_by_value_range(df_rarest, COLUMN_TIME, value_range=(start_time, end_time))
    df_rarest_list.append(df_rarest_around_peak)

## Download granules

Failed downloads are logged and stored as `None`.

In [ ]:
# Specify the product and product type
product = "2A-DPR"  # 2A-PR
product_type = "RS"
storage = "GES_DISC"
# Specify the version
version = 7

ds_dpr_list = []
for i in peaks_indices:
    peak = df_peaks[i]

    # Specify the time period you are interested in
    timedelta = datetime.timedelta(minutes=1.2)
    start_time = peak["time"].item() - timedelta - datetime.timedelta(minutes=1.1)
    end_time = peak["time"].item() + timedelta - datetime.timedelta(minutes=1.1)

    ds = None
    try:
        # Download the data
        gpm.download(
            product=product,
            product_type=product_type,
            version=version,
            start_time=start_time,
            end_time=end_time,
            storage=storage,
            force_download=False,
            verbose=True,
            progress_bar=True,
            check_integrity=False,
        )

        # Load the 2A-DPR dataset
        ds = gpm.open_dataset(
            product=product,
            product_type=product_type,
            version=version,
            start_time=start_time,
            end_time=end_time,
            variables=[
                "zFactorFinal",
                "zFactorFinalNearSurface",
                "zFactorMeasured",
                # 'flagSigmaZeroSaturation',
                # 'sigmaZeroCorrected',
                'sigmaZeroMeasured',
                # 'sigmaZeroNPCorrected',
                # 'sigmaZeroProfile',
                "precipRateNearSurface",
                "flagPrecip",
            ]
        )
        ds = ds.compute()
    except Exception as e:
        logging.warning(f"Error for peak {i} with longitude {peak[COLUMN_LON].item():.2f} and latitude {peak[COLUMN_LAT].item():.2f}: {e}")
    finally:
        ds_dpr_list.append(ds)

In [ ]:
# Specify the product and product type
product = "1C-GMI-R"
product_2a = "2A-GMI"

product_type = "RS"
storage = "GES_DISC"
# Specify the version
version = 7

ds_pmw_list = []
for i in peaks_indices:
    peak = df_peaks[i]

    # Specify the time period you are interested in
    timedelta = datetime.timedelta(minutes=1)
    start_time = peak["time"].item() - timedelta
    end_time = peak["time"].item() + timedelta

    ds_pmw = None
    try:
        # Download the data
        for p in [product, product_2a]:
            gpm.download(
                product=p,
                product_type=product_type,
                version=version,
                start_time=start_time,
                end_time=end_time,
                storage=storage,
                force_download=False,
                verbose=True,
                progress_bar=True,
                check_integrity=False,
            )

        # Load the 1C-GMI-R dataset
        ds_pmw = gpm.open_dataset(
            product=product,
            product_type=product_type,
            version=version,
            start_time=start_time,
            end_time=end_time,
        )
        ds_temp2m = gpm.open_dataset(
            product=product_2a,
            product_type=product_type,
            version=version,
            start_time=start_time,
            end_time=end_time,
            variables=["temp2mIndex", "surfaceTypeIndex"]
        )
        ds_pmw = ds_pmw.assign(t2m=ds_temp2m["temp2mIndex"])
        ds_pmw = ds_pmw.assign(surfaceTypeIndex=ds_temp2m["surfaceTypeIndex"])
        ds_pmw = ds_pmw.compute()
    except Exception as e:
        logging.warning(f"Error for peak {i} with longitude {peak[COLUMN_LON].item():.2f} and latitude {peak[COLUMN_LAT].item():.2f}: {e}")
    finally:
        ds_pmw_list.append(ds_pmw)

In [ ]:
ds_pesca_list = [None if ds_pmw is None else ds_pmw.gpm.retrieve("PESCA") for ds_pmw in ds_pmw_list]

## Plots

Per peak: DPR variables (Ka band, profiles at range bins 100–170); GMI features alone, with the rarest observations (magenta) and with DPR near-surface precipitation; PESCA and surface type.

In [ ]:
range_values = [100, 125, 150, 170]

for idx, i in enumerate(peaks_indices):
    peak = df_peaks[i]
    ds = ds_dpr_list[idx]

    if ds is None:
        continue

    variables_to_plot = ["sigmaZeroMeasured", "zFactorMeasured", "precipRateNearSurface", "zFactorFinalNearSurface", "flagPrecip"]
    data_arrays_to_plot = []
    titles = []

    for var in variables_to_plot:
        da = ds[var]
        if var == "zFactorMeasured":
            da = da.where(da > 18)

        if gpm.checks.has_frequency_dim(da):
            da = da.sel(radar_frequency="Ka")

        sel_dicts = [{}]
        if gpm.checks.has_vertical_dim(da):
            sel_dicts_new = []
            for sel_dict in sel_dicts:
                for range_value in range_values:
                    sel_dict_new = sel_dict.copy()
                    sel_dict_new["range"] = range_value
                    sel_dicts_new.append(sel_dict_new)
            sel_dicts = sel_dicts_new

        for sel_dict in sel_dicts:
            data_arrays_to_plot.append(da.sel(**sel_dict))
            titles.append(f"{var} ({sel_dict})")

    projection, figsize = _get_projection(peak)
    figsize = (figsize[0], len(data_arrays_to_plot) * figsize[1])

    fig, axes = plt.subplots(nrows=len(data_arrays_to_plot), ncols=1, subplot_kw={"projection": projection}, figsize=figsize)

    for ax, da, title in zip(axes, data_arrays_to_plot, titles):
        da.gpm.plot_map(ax=ax)
        ax.set_title(title)

    plt.tight_layout()
    plt.savefig(images_dir / f"lon={peak['lon'].item():.2f}_lat={peak['lat'].item():.2f}_dpr.png")

In [ ]:
for idx, i in enumerate(peaks_indices):
    peak = df_peaks[i]
    ds_pmw = ds_pmw_list[idx]
    ds_dpr = ds_dpr_list[idx]
    df_rarest_around_peak = df_rarest_list[idx]

    if ds_pmw is None or ds_dpr is None:
        continue

    ds_tc = ds_pmw["Tc"].to_dataset(dim="pmw_frequency", promote_attrs=True)
    ds_tc = ds_tc.rename({var: "Tc_" + var for var in ds_tc.data_vars})
    ds_out = transform_ds(ds_tc, drop=True)

    variables_to_plot = ["PD_19", "PD_37", "Tc_37V_Tc_19V", "Tc_89V_Tc_37V","Tc_37V"]

    projection, figsize = _get_projection(peak)
    ncols = 3
    figsize = (ncols * figsize[0], len(variables_to_plot) * figsize[1])

    fig, axes = plt.subplots(nrows=len(variables_to_plot), ncols=ncols, subplot_kw={"projection": projection}, figsize=figsize)

    for axes_row, var in zip(axes, variables_to_plot):
        colorbar_name = "PD" if var.find("Tc") != var.rfind("Tc") or var.find("Tc") == -1 else "Tc"
        plot_kwargs, cbar_kwargs = gpm.get_plot_kwargs(colorbar_name)
        if colorbar_name == "PD":
            norm = plot_kwargs["norm"]
            norm.vmin = None
            plot_kwargs["norm"] = norm

        for ax in axes_row:
            da = ds_out[var]
            da.gpm.plot_map(ax=ax, cbar_kwargs=cbar_kwargs, **plot_kwargs)
            ax.set_title(var)
        axes_row[1].scatter(df_rarest_around_peak[COLUMN_LON], df_rarest_around_peak[COLUMN_LAT], c="magenta", marker="X", s=1, transform=ccrs.Geodetic())
        ds_dpr["precipRateNearSurface"].gpm.plot_map(ax=axes_row[2])

    plt.tight_layout()

    plt.savefig(images_dir / f"lon={peak['lon'].item():.2f}_lat={peak['lat'].item():.2f}_pmw.png")

In [ ]:
for idx, i in enumerate(peaks_indices):
    peak = df_peaks[i]
    ds_pmw = ds_pmw_list[idx]
    da_pesca = ds_pesca_list[idx].isel(pmw_frequency=0)

    projection, figsize = _get_projection(peak)
    figsize = (2 * figsize[0], figsize[1])

    fig, (ax1, ax2) = plt.subplots(ncols=2, subplot_kw={"projection": projection}, figsize=figsize)

    da_pesca.gpm.plot_map(ax=ax1)
    ax1.set_title("PESCA")

    ds_pmw["surfaceTypeIndex"].gpm.plot_map(ax=ax2)
    ax2.set_title("surfaceTypeIndex")

    plt.tight_layout()

    plt.savefig(images_dir / f"lon={peak['lon'].item():.2f}_lat={peak['lat'].item():.2f}_pesca.png")
    plt.show()